In [0]:
sales_data = [
    (1, "Kochi", "2024-01-01", 50000),
    (2, "Kochi", "2024-01-03", 30000),
    (3, "Kochi", "2024-01-05", 40000),
    (4, "Chennai", "2024-01-01", 45000),
    (5, "Chennai", "2024-01-04", 50000),
    (6, "Chennai", "2024-01-06", 35000),
    (7, "Bangalore", "2024-01-02", 60000),
    (8, "Bangalore", "2024-01-05", 65000),
    (9, "Bangalore", "2024-01-07", 55000),
    (10, "Hyderabad", "2024-01-01", 40000),
    (11, "Hyderabad", "2024-01-03", 45000),
    (12, "Hyderabad", "2024-01-06", 50000)
]
 
columns = ["order_id", "city", "order_date", "revenue"]
 
df_day9 = spark.createDataFrame(sales_data, columns)

# Level 1: lag / lead
- Q1 Add: previous revenue using lag()


In [0]:
from pyspark.sql.functions import lag
from pyspark.sql.window import Window
df_day9.withColumn("previous_revenue",lag("revenue",1).over(Window.partitionBy("city").orderBy("order_date"))).show()

+--------+---------+----------+-------+----------------+
|order_id|     city|order_date|revenue|previous_revenue|
+--------+---------+----------+-------+----------------+
|       7|Bangalore|2024-01-02|  60000|            NULL|
|       8|Bangalore|2024-01-05|  65000|           60000|
|       9|Bangalore|2024-01-07|  55000|           65000|
|       4|  Chennai|2024-01-01|  45000|            NULL|
|       5|  Chennai|2024-01-04|  50000|           45000|
|       6|  Chennai|2024-01-06|  35000|           50000|
|      10|Hyderabad|2024-01-01|  40000|            NULL|
|      11|Hyderabad|2024-01-03|  45000|           40000|
|      12|Hyderabad|2024-01-06|  50000|           45000|
|       1|    Kochi|2024-01-01|  50000|            NULL|
|       2|    Kochi|2024-01-03|  30000|           50000|
|       3|    Kochi|2024-01-05|  40000|           30000|
+--------+---------+----------+-------+----------------+



- Q2 Add: next revenue using lead()

In [0]:
from pyspark.sql.functions import lead, col
from pyspark.sql.window import Window
df_day9.withColumn("next_revenue",lead("revenue",1).over(Window.partitionBy("city").orderBy("order_date"))).show()

+--------+---------+----------+-------+------------+
|order_id|     city|order_date|revenue|next_revenue|
+--------+---------+----------+-------+------------+
|       7|Bangalore|2024-01-02|  60000|       65000|
|       8|Bangalore|2024-01-05|  65000|       55000|
|       9|Bangalore|2024-01-07|  55000|        NULL|
|       4|  Chennai|2024-01-01|  45000|       50000|
|       5|  Chennai|2024-01-04|  50000|       35000|
|       6|  Chennai|2024-01-06|  35000|        NULL|
|      10|Hyderabad|2024-01-01|  40000|       45000|
|      11|Hyderabad|2024-01-03|  45000|       50000|
|      12|Hyderabad|2024-01-06|  50000|        NULL|
|       1|    Kochi|2024-01-01|  50000|       30000|
|       2|    Kochi|2024-01-03|  30000|       40000|
|       3|    Kochi|2024-01-05|  40000|        NULL|
+--------+---------+----------+-------+------------+



# Level 2: Comparison
- Q3 Create: revenue difference (current - previous)

In [0]:
from pyspark.sql.functions import lag, col
from pyspark.sql.window import Window
df_day9.withColumn("revenue_diff",col("revenue") - lag("revenue",1).over(Window.partitionBy("city").orderBy("order_date"))).show()

+--------+---------+----------+-------+------------+
|order_id|     city|order_date|revenue|revenue_diff|
+--------+---------+----------+-------+------------+
|       7|Bangalore|2024-01-02|  60000|        NULL|
|       8|Bangalore|2024-01-05|  65000|        5000|
|       9|Bangalore|2024-01-07|  55000|      -10000|
|       4|  Chennai|2024-01-01|  45000|        NULL|
|       5|  Chennai|2024-01-04|  50000|        5000|
|       6|  Chennai|2024-01-06|  35000|      -15000|
|      10|Hyderabad|2024-01-01|  40000|        NULL|
|      11|Hyderabad|2024-01-03|  45000|        5000|
|      12|Hyderabad|2024-01-06|  50000|        5000|
|       1|    Kochi|2024-01-01|  50000|        NULL|
|       2|    Kochi|2024-01-03|  30000|      -20000|
|       3|    Kochi|2024-01-05|  40000|       10000|
+--------+---------+----------+-------+------------+



- Q4 Find: rows where revenue increased

In [0]:
from pyspark.sql.functions import lag, col
from pyspark.sql.window import Window
df_day9.withColumn("prev_revenue", lag("revenue", 1).over(Window.partitionBy("city").orderBy("order_date"))).filter(col("revenue") > col("prev_revenue")).show()


+--------+---------+----------+-------+------------+
|order_id|     city|order_date|revenue|prev_revenue|
+--------+---------+----------+-------+------------+
|       8|Bangalore|2024-01-05|  65000|       60000|
|       5|  Chennai|2024-01-04|  50000|       45000|
|      11|Hyderabad|2024-01-03|  45000|       40000|
|      12|Hyderabad|2024-01-06|  50000|       45000|
|       3|    Kochi|2024-01-05|  40000|       30000|
+--------+---------+----------+-------+------------+



# Level 3: Running Calculations
- Q5 Calculate: running total revenue per city


In [0]:
from pyspark.sql.functions import sum
df_day9.withColumn("running_revenue", sum("revenue").over(Window.partitionBy("city").orderBy("order_date"))).show()

+--------+---------+----------+-------+---------------+
|order_id|     city|order_date|revenue|running_revenue|
+--------+---------+----------+-------+---------------+
|       7|Bangalore|2024-01-02|  60000|          60000|
|       8|Bangalore|2024-01-05|  65000|         125000|
|       9|Bangalore|2024-01-07|  55000|         180000|
|       4|  Chennai|2024-01-01|  45000|          45000|
|       5|  Chennai|2024-01-04|  50000|          95000|
|       6|  Chennai|2024-01-06|  35000|         130000|
|      10|Hyderabad|2024-01-01|  40000|          40000|
|      11|Hyderabad|2024-01-03|  45000|          85000|
|      12|Hyderabad|2024-01-06|  50000|         135000|
|       1|    Kochi|2024-01-01|  50000|          50000|
|       2|    Kochi|2024-01-03|  30000|          80000|
|       3|    Kochi|2024-01-05|  40000|         120000|
+--------+---------+----------+-------+---------------+



- Q6 Calculate: cumulative average revenue per city

In [0]:
from pyspark.sql.functions import avg
df_day9.withColumn("cummulative_avg_revenue", avg("revenue").over(Window.partitionBy("city").orderBy("order_date"))).show()

+--------+---------+----------+-------+-----------------------+
|order_id|     city|order_date|revenue|cummulative_avg_revenue|
+--------+---------+----------+-------+-----------------------+
|       7|Bangalore|2024-01-02|  60000|                60000.0|
|       8|Bangalore|2024-01-05|  65000|                62500.0|
|       9|Bangalore|2024-01-07|  55000|                60000.0|
|       4|  Chennai|2024-01-01|  45000|                45000.0|
|       5|  Chennai|2024-01-04|  50000|                47500.0|
|       6|  Chennai|2024-01-06|  35000|     43333.333333333336|
|      10|Hyderabad|2024-01-01|  40000|                40000.0|
|      11|Hyderabad|2024-01-03|  45000|                42500.0|
|      12|Hyderabad|2024-01-06|  50000|                45000.0|
|       1|    Kochi|2024-01-01|  50000|                50000.0|
|       2|    Kochi|2024-01-03|  30000|                40000.0|
|       3|    Kochi|2024-01-05|  40000|                40000.0|
+--------+---------+----------+-------+-

# Level 4: Trend Analysis
- Q7 Find: highest revenue jump per city


In [0]:
from pyspark.sql.functions import lag, col, max
from pyspark.sql.window import Window

df_day9.withColumn("revenue_jump", col("revenue") - lag("revenue", 1).over(Window.partitionBy("city").orderBy("order_date"))).groupBy("city").agg(max("revenue_jump").alias("max_revenue_jump")).show()

+---------+----------------+
|     city|max_revenue_jump|
+---------+----------------+
|Bangalore|            5000|
|  Chennai|            5000|
|Hyderabad|            5000|
|    Kochi|           10000|
+---------+----------------+



- Q8 Find: biggest drop in revenue

In [0]:
from pyspark.sql.functions import lag, col, min
from pyspark.sql.window import Window

df_day9.withColumn("revenue_drop", col("revenue") - lag("revenue", 1).over(Window.partitionBy("city").orderBy("order_date"))
).groupBy("city").agg(min("revenue_drop").alias("min_revenue_drop")).show()

+---------+----------------+
|     city|min_revenue_drop|
+---------+----------------+
|Bangalore|          -10000|
|  Chennai|          -15000|
|Hyderabad|            5000|
|    Kochi|          -20000|
+---------+----------------+



#  Level 5: Challenge 
- Q9 Calculate: percentage growth from previous order


In [0]:
from pyspark.sql.functions import lag, col, round
from pyspark.sql.window import Window

df_day9.withColumn("prev_revenue", lag("revenue", 1).over(Window.partitionBy("city").orderBy("order_date"))).withColumn(
    "percentage_growth",round(((col("revenue") - col("prev_revenue")) / col("prev_revenue")) * 100, 2)).show()

+--------+---------+----------+-------+------------+-----------------+
|order_id|     city|order_date|revenue|prev_revenue|percentage_growth|
+--------+---------+----------+-------+------------+-----------------+
|       7|Bangalore|2024-01-02|  60000|        NULL|             NULL|
|       8|Bangalore|2024-01-05|  65000|       60000|             8.33|
|       9|Bangalore|2024-01-07|  55000|       65000|           -15.38|
|       4|  Chennai|2024-01-01|  45000|        NULL|             NULL|
|       5|  Chennai|2024-01-04|  50000|       45000|            11.11|
|       6|  Chennai|2024-01-06|  35000|       50000|            -30.0|
|      10|Hyderabad|2024-01-01|  40000|        NULL|             NULL|
|      11|Hyderabad|2024-01-03|  45000|       40000|             12.5|
|      12|Hyderabad|2024-01-06|  50000|       45000|            11.11|
|       1|    Kochi|2024-01-01|  50000|        NULL|             NULL|
|       2|    Kochi|2024-01-03|  30000|       50000|            -40.0|
|     

- Q10 Find: city with most consistent growth

In [0]:
from pyspark.sql.functions import lag, col, stddev,desc,asc
from pyspark.sql.window import Window

df= df_day9.withColumn("prev_revenue", lag("revenue", 1).over(Window.partitionBy("city").orderBy("order_date"))).withColumn("percentage_growth", ((col("revenue") - col("prev_revenue")) / col("prev_revenue")) * 100).filter(col("percentage_growth").isNotNull())

df_cons = df.groupBy("city").agg(stddev("percentage_growth").alias("growth_stddev")).orderBy(asc("growth_stddev")).show(1)


+---------+------------------+
|     city|     growth_stddev|
+---------+------------------+
|Hyderabad|0.9820927516479829|
+---------+------------------+
only showing top 1 row
